# Demo 04 : A2A on Agent Server

Run the cells in order and watch.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · a2a-on-agent-server

We discover the Deep Agent supervisor's card, then call the stock graph through A2A. langgraph-api 0.16.7 requires assistant UUIDs for A2A routes and the agent card; graph names are rejected. Module 06 covers cross-server communication.

In [2]:
import json
from pathlib import Path

In [3]:
config = {"dependencies": ["."], "graphs": {"service": "./graphs.py:service",
          "researcher": "./graphs.py:researcher", "supervisor": "./graphs.py:supervisor"},
          "env": "../../.env"}
Path("langgraph.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))

{
  "dependencies": [
    "."
  ],
  "graphs": {
    "service": "./graphs.py:service",
    "researcher": "./graphs.py:researcher",
    "supervisor": "./graphs.py:supervisor"
  },
  "env": "../../.env"
}


In [4]:
import atexit
import os
import signal
import socket
import subprocess
import time
from pathlib import Path

import httpx
from langgraph_sdk import get_sync_client

HERE = Path.cwd()
servers = []


def port_open(port):
    with socket.socket() as probe:
        probe.settimeout(0.5)
        return probe.connect_ex(("127.0.0.1", port)) == 0


def wait_port_free(port, seconds=60):
    for _ in range(seconds * 2):
        if not port_open(port):
            return
        time.sleep(0.5)
    raise TimeoutError(f"Port {port} stayed busy after stop.")


def stop_server(proc):
    if proc is None:
        return
    port = getattr(proc, "course_port", None)
    was_running = proc.poll() is None
    if was_running:
        proc.send_signal(signal.CTRL_BREAK_EVENT if os.name == "nt" else signal.SIGINT)
        try:
            proc.wait(timeout=25)
        except subprocess.TimeoutExpired:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"],
                               capture_output=True, check=False)
            else:
                proc.kill()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
    if was_running and port is not None:
        wait_port_free(port, seconds=90)


def start_server(port):
    if port not in (2134, 2135):
        raise ValueError("Use a module 09 port.")
    wait_port_free(port, seconds=90)
    log = (HERE / f"server-{port}.log").open("w", encoding="utf-8")
    env = dict(os.environ, PYTHONIOENCODING="utf-8", LANGGRAPH_CLI_NO_ANALYTICS="1")
    env["LANGSMITH_TRACING"] = "false"
    proc = subprocess.Popen(
        ["uv", "run", "langgraph", "dev", "--no-browser", "--no-reload", "--port", str(port),
         "--n-jobs-per-worker", "8", "--studio-url", "https://eu.smith.langchain.com"],
        cwd=HERE, env=env, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)
    proc.course_port = port
    log.close()
    servers.append(proc)
    atexit.register(stop_server, proc)
    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError((HERE / f"server-{port}.log").read_text(encoding="utf-8")[-5000:])
        try:
            response = httpx.get(f"http://127.0.0.1:{port}/ok", timeout=1)
            if response.is_success:
                return proc
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_server(proc)
    raise TimeoutError("Agent Server did not become ready.")


def connect(port):
    return get_sync_client(url=f"http://127.0.0.1:{port}", api_key=None, timeout=240)


def clear_dev_storage():
    import shutil
    target = HERE / ".langgraph_api"
    if target.exists():
        shutil.rmtree(target)
        print("cleared :", target.name)

In [5]:
from uuid import uuid4

clear_dev_storage()
server = start_server(2135)
client = connect(2135)
URL = "http://127.0.0.1:2135"
assistants = {a["graph_id"]: a["assistant_id"] for a in client.assistants.search()}
print("assistants :", assistants)
card = httpx.get(URL + "/.well-known/agent-card.json",
                 params={"assistant_id": assistants["supervisor"]}).json()
print("Deep Agent card :", card["name"], card["supportedInterfaces"])
response = httpx.post(URL + "/a2a/" + assistants["service"], timeout=240, json={
    "jsonrpc": "2.0", "id": "stock", "method": "SendMessage", "params": {
        "message": {"role": "ROLE_USER", "messageId": str(uuid4()),
                    "parts": [{"text": "How many CE-CF-02 fans are in stock and where?"}]},
        "configuration": {"blocking": True, "historyScope": "task"}}})
response.raise_for_status()
body = response.json()
assert "error" not in body, body
task = body["result"]["task"]
print("task :", task["id"], task.get("status"))
print("context ID :", task["contextId"])
print("artifacts :", json.dumps(task.get("artifacts", []), ensure_ascii=False))
print("history :", json.dumps(task.get("history", []), ensure_ascii=False))

assistants : {'supervisor': '3dba61cc-373b-5094-96d6-8331cb40a952', 'researcher': 'c926ac5a-b04e-5949-878a-8e4830d4338b', 'service': '32ac0052-56bf-5251-9244-9f99ebc609f9'}
Deep Agent card : supervisor [{'url': 'http://127.0.0.1:2135/a2a/3dba61cc-373b-5094-96d6-8331cb40a952', 'protocolBinding': 'JSONRPC', 'protocolVersion': '1.0'}]


task : 8b5998a5-4f09-4eed-a820-a8a7827408c9:01a17b67-be6b-7c41-8ec4-c5c9883dab3b {'state': 'TASK_STATE_COMPLETED', 'timestamp': '2026-10-27T03:29:26.009595+00:00'}
context ID : 8b5998a5-4f09-4eed-a820-a8a7827408c9
artifacts : [{"artifactId": "2b9512df-743d-4964-b508-3ca5c5bb9546", "name": "Assistant Response", "description": "Response from assistant 32ac0052-56bf-5251-9244-9f99ebc609f9", "parts": [{"kind": "text", "text": "There are 6 CE-CF-02 fans in stock at Rietpoort."}]}]
history : [{"kind": "message", "role": "ROLE_USER", "parts": [{"kind": "text", "text": "How many CE-CF-02 fans are in stock and where?"}], "messageId": "e451c51e-d58e-49a2-b1b8-3ac34f3236ec", "taskId": "8b5998a5-4f09-4eed-a820-a8a7827408c9:01a17b67-be6b-7c41-8ec4-c5c9883dab3b", "contextId": "8b5998a5-4f09-4eed-a820-a8a7827408c9"}, {"kind": "message", "role": "ROLE_AGENT", "parts": [{"kind": "data", "data": {"tool_results": [{"toolCallId": "call_MTpG8Vm2Whd6gplPz34Bvi2b", "content": "[[\"CE-CF-02\", 6, \"Rietpoort\

In [6]:
for proc in servers:
    stop_server(proc)
print("servers stopped :", all(proc.poll() is not None for proc in servers))

servers stopped : True
